# 30 — Matryoshka WJ MLP 512

A modern representation-compression variant. The model outputs 512 dimensions, but the loss is applied to prefixes 64/128/256/512 so shorter prefixes remain useful.

This is a good paper-extension experiment: it can show adaptive WJ-sensitive embeddings at multiple vector sizes.


In [1]:
import os
import random
import sys
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

sys.path.append('/raid/ruban/hpmlproj/term_project')
from sota_experiment_common import (
    cleanup,
    eval_recall,
    l1_simplex,
    load_dataset,
    nmslib_neighbors,
    rerank_raw_wj_numpy,
    save_result,
)

# Edit here
dataset_name = "10k"
out_dim = 512
device_str = "cuda:0"
device = torch.device(device_str if torch.cuda.is_available() else "cpu")
THREADS = 32
seed = 42
batch_size = 512
epochs = 30
lr = 1e-3
weight_decay = 1e-4
candidate_ks = [500, 1000] if dataset_name == "10k" else [1000, 2000]
run_rerank = True

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
print(f"device={device}")

METHOD_NAME = "matryoshka_mlp_wj_512"
NOTEBOOK_NAME = "30_matryoshka_mlp_wj_512.ipynb"
OUT_PATH = "/tmp/results_sota_matryoshka_mlp_wj_512.pkl"
CKPT_PATH = "/tmp/best_sota_matryoshka_mlp_wj_512.pt"
max_pos = 30
margin = 0.3
prefix_dims = [64, 128, 256, 512]


device=cuda:0


In [2]:
qt, gt, query_start, corpus_qt, query_qt, corpus_sums = load_dataset(dataset_name)
qt_norm = l1_simplex(qt.copy())


dataset=10k | qt=(10000, 18499) | corpus=(8000, 18499) | queries=(2000, 18499)


In [3]:
def wj_torch(a, b):
    mins = torch.minimum(a, b).sum(dim=-1)
    maxs = torch.maximum(a, b).sum(dim=-1).clamp(min=1e-10)
    return mins / maxs

class PairDataset(Dataset):
    def __init__(self, qt, gt, query_start, max_pos=30):
        self.vecs = torch.tensor(qt, dtype=torch.float32)
        self.pairs = []
        for qid, neighbors in gt.items():
            for nid in neighbors[:max_pos]:
                if qid >= query_start and nid < query_start:
                    self.pairs.append((qid, nid))
        random.shuffle(self.pairs)
        print(f"pairs={len(self.pairs):,}")
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        qid, pid = self.pairs[idx]
        rid = random.randrange(0, query_start)
        return self.vecs[qid], self.vecs[pid], self.vecs[rid]

def embed_all(model, qt, batch_size=512):
    model.eval()
    out = []
    with torch.no_grad():
        for start in range(0, len(qt), batch_size):
            x = torch.tensor(qt[start:start + batch_size], dtype=torch.float32, device=device)
            out.append(model.encode(x).detach().cpu().numpy().astype(np.float32))
    return np.vstack(out)

def eval_embeddings(embs, method_name, out_path, notebook_name):
    corpus_embs = embs[:query_start]
    query_embs = embs[query_start:]
    max_k = max(max(candidate_ks), 500)
    print(f"embs={embs.shape} | mem={corpus_embs.nbytes/1024**2:.1f} MB")
    nbrs, info = nmslib_neighbors(corpus_embs, query_embs, space="WeightedJaccard", k=max_k, threads=THREADS)
    metrics = {**eval_recall(gt, nbrs, query_start, max_k), **info, "dim": out_dim, "vec_mb": corpus_embs.nbytes/1024**2}
    for k, v in metrics.items():
        if isinstance(k, int): print(f"R@{k:<4} = {v:.4f}")
    print(f"QPS={metrics['qps']:.1f}")
    save_result(out_path, dataset_name, method_name, metrics, meta={"notebook": notebook_name})
    if run_rerank:
        for ck in candidate_ks:
            cand, cand_info = nmslib_neighbors(corpus_embs, query_embs, space="WeightedJaccard", k=ck, threads=THREADS)
            t0 = time.time()
            rr = rerank_raw_wj_numpy(query_qt, cand, corpus_qt, corpus_sums, top_k=ck, batch_size=8)
            qps_total = len(query_qt) / max(time.time() - t0 + len(query_qt)/max(cand_info['qps'], 1e-9), 1e-9)
            rr_metrics = {**eval_recall(gt, rr, query_start, ck), "qps": qps_total, "candidate_k": ck}
            key = f"{method_name}_rerank_{ck}"
            for k, v in rr_metrics.items():
                if isinstance(k, int): print(f"{key} R@{k:<4} = {v:.4f}")
            save_result(out_path, dataset_name, key, rr_metrics, meta={"notebook": notebook_name})


In [4]:
class MatryoshkaWJMLP(nn.Module):
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False), nn.BatchNorm1d(out_dim),
        )
    def encode(self, x):
        z = F.relu(self.net(x))
        return z / z.sum(dim=1, keepdim=True).clamp(min=1e-10)
    def forward(self, x):
        return self.encode(x)

def prefix_norm(z, d):
    p = z[:, :d]
    return p / p.sum(dim=1, keepdim=True).clamp(min=1e-10)

model = MatryoshkaWJMLP(qt.shape[1], out_dim).to(device)
dataset = PairDataset(qt_norm, gt, query_start, max_pos=max_pos)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
best = float('inf')
for epoch in range(1, epochs + 1):
    model.train(); total = 0; steps = 0
    for a, p, r in loader:
        a = a.to(device, non_blocking=True); p = p.to(device, non_blocking=True); r = r.to(device, non_blocking=True)
        z = model(torch.cat([a, p, r], dim=0))
        za, zp, zr = z.chunk(3, dim=0)
        losses = []
        for d in prefix_dims:
            pa, pp, pr = prefix_norm(za, d), prefix_norm(zp, d), prefix_norm(zr, d)
            sim_ap = wj_torch(pa, pp)
            sim_ar = wj_torch(pa, pr)
            losses.append(F.relu(sim_ar - sim_ap + margin).mean())
        loss = sum(losses) / len(losses)
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        total += float(loss.detach()); steps += 1
    avg = total / max(steps, 1)
    if avg < best:
        best = avg; torch.save(model.state_dict(), CKPT_PATH)
    if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
        print(f"epoch {epoch:02d}/{epochs} loss={avg:.4f}")
print(f"best={best:.4f} saved {CKPT_PATH}")


pairs=46,722
epoch 01/30 loss=0.0106
epoch 05/30 loss=0.0076
epoch 10/30 loss=0.0069
epoch 15/30 loss=0.0069
epoch 20/30 loss=0.0067
epoch 25/30 loss=0.0062
epoch 30/30 loss=0.0061
best=0.0060 saved /tmp/best_sota_matryoshka_mlp_wj_512.pt


In [5]:
model.load_state_dict(torch.load(CKPT_PATH, map_location=device, weights_only=True))
full_embs = embed_all(model, qt_norm, batch_size=512)
for d in prefix_dims:
    embs = l1_simplex(full_embs[:, :d].copy())
    name = f"{METHOD_NAME}_d{d}"
    print(f"\n=== {name} ===")
    old_dim = out_dim
    out_dim = d
    eval_embeddings(embs, name, OUT_PATH, NOTEBOOK_NAME)
    out_dim = old_dim
cleanup()



=== matryoshka_mlp_wj_512_d64 ===
embs=(10000, 64) | mem=2.0 MB



0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

R@10   = 0.4425
R@50   = 0.6619
R@100  = 0.7531
R@500  = 0.9305
QPS=24913.7
saved matryoshka_mlp_wj_512_d64 -> /tmp/results_sota_matryoshka_mlp_wj_512.pkl



0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

matryoshka_mlp_wj_512_d64_rerank_500 R@10   = 0.9966
matryoshka_mlp_wj_512_d64_rerank_500 R@50   = 0.9980
matryoshka_mlp_wj_512_d64_rerank_500 R@100  = 0.9959
matryoshka_mlp_wj_512_d64_rerank_500 R@500  = 0.9306
saved matryoshka_mlp_wj_512_d64_rerank_500 -> /tmp/results_sota_matryoshka_mlp_wj_512.pkl


...

R@10   = 0.4548
R@50   = 0.6716
R@100  = 0.7604
R@500  = 0.9317
QPS=18646.7
saved matryoshka_mlp_wj_512_d512 -> /tmp/results_sota_matryoshka_mlp_wj_512.pkl



0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

matryoshka_mlp_wj_512_d512_rerank_500 R@10   = 0.9966
matryoshka_mlp_wj_512_d512_rerank_500 R@50   = 0.9980
matryoshka_mlp_wj_512_d512_rerank_500 R@100  = 0.9961
matryoshka_mlp_wj_512_d512_rerank_500 R@500  = 0.9315
saved matryoshka_mlp_wj_512_d512_rerank_500 -> /tmp/results_sota_matryoshka_mlp_wj_512.pkl



0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

matryoshka_mlp_wj_512_d512_rerank_1000 R@10   = 0.9966
matryoshka_mlp_wj_512_d512_rerank_1000 R@50   = 0.9980
matryoshka_mlp_wj_512_d512_rerank_1000 R@100  = 0.9962
matryoshka_mlp_wj_512_d512_rerank_1000 R@500  = 0.9346
saved matryoshka_mlp_wj_512_d512_rerank_1000 -> /tmp/results_sota_matryoshka_mlp_wj_512.pkl
